# Task 2：ResNet34 全量训练与提交

使用全部 **28,275 张带标签训练图片**，不划验证集；测试图片不参与训练。训练方案由此前验证实验选定，不再依据本轮训练准确率挑模型。

- 从 ImageNet 预训练 ResNet34 重新开始，不接着旧验证模型追加训练。
- 固定训练 **14 轮**，对应之前最佳原图验证准确率所在轮次。采用原先 `T_max=25` 的余弦学习率调度，只运行前 14 轮，避免把调度压缩成另一套方案。
- 输入 224、batch=128、lr=3e-4、weight_decay=1e-4、Dropout=0、标签平滑=0，增强保持不变。全量训练每轮更新次数更多，并不保证最优轮次仍为 14 或榜单一定提分。
- 用第 14 轮最终权重及水平翻转 TTA 预测，官方编码为 Angry=0、Fear=1、Happy=2、Sad=3、Surprise=4、Neutral=5。
- 输出目录独立为 `/kaggle/working/task2_fulltrain_resnet34_seed42`，不覆盖旧消融实验。

**Kaggle 操作**：开启 GPU 和 Internet，只添加比赛数据即可，从头运行。原有第三轮备份不是必需的。

每轮结束保存 `last.pt`（模型、优化器、调度器和 AMP 状态）。同一会话再次运行会自动续训；新会话需将完整全量训练备份作为 Input 添加，程序会自动恢复唯一匹配的备份。正在执行的未完成轮次会重跑。最后的备份单元格可在手动停止训练后单独运行；仍须下载备份，working 目录不是永久存储。

本轮不产生验证准确率，训练准确率不能当成泛化成绩。已有验证结果与当前榜单 68.4% 留作此前方案的记录。


In [ ]:
import hashlib
import json
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
try:
    from IPython.display import FileLink, display
except ImportError:
    def display(*items):
        for item in items:
            print(item, flush=True)
    def FileLink(path):
        return str(path)

SEED = 42
EPOCHS = 14
SCHEDULER_T_MAX = 25  # 沿用选模实验的逐轮学习率，不压缩为 14
BATCH_SIZE = 128
IMAGE_SIZE = 224
LR = 3e-4
WEIGHT_DECAY = 1e-4
WORKERS = 0
PRETRAINED = True
EXPECTED_TRAIN = 28275
EXPECTED_TEST = 7065
DATA_DIR = None       # 有多份比赛数据时，填写含 train/ 与 test/ 的目录
RESUME_DIR = None     # 有多份全量备份时，填写目标备份目录
CLASSES = ["Angry", "Fear", "Happy", "Neutral", "Sad", "Surprise"]
LABEL_MAP = {"Angry": 0, "Fear": 1, "Happy": 2, "Sad": 3, "Surprise": 4, "Neutral": 5}
ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
OUT = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()) / f"task2_fulltrain_resnet34_seed{SEED}"
OUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(4)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("设备:", device, "| 输出目录:", OUT)


## 1. 读取全部带标签图片
检查训练/测试文件名、官方提交模板和数据指纹。所有训练图片均纳入，不筛选或删除重复样本，不从测试图片提取标签。


In [ ]:
if DATA_DIR is None:
    candidates = [p.parent for p in ROOT.rglob("train") if p.is_dir()
                  and (p.parent / "test").is_dir()
                  and sorted(d.name for d in p.iterdir() if d.is_dir()) == CLASSES]
    if len(candidates) != 1:
        raise ValueError(f"找到 {len(candidates)} 份比赛数据，请设置 DATA_DIR。")
    DATA_DIR = candidates[0]
DATA_DIR = Path(DATA_DIR)
paths, labels, rows = [], [], []
for label, name in enumerate(CLASSES):
    class_paths = sorted((DATA_DIR / "train" / name).glob("*.jpg"))
    if not class_paths:
        raise ValueError(f"类别 {name} 没有图片。")
    for path in class_paths:
        paths.append(path)
        labels.append(label)
        rows.append(dict(path=path.relative_to(DATA_DIR).as_posix(), label=name,
                         file_sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
labels = np.array(labels, dtype=np.int64)
manifest = pd.DataFrame(rows)
manifest_text = manifest.to_csv(index=False, lineterminator="\n")
DATA_HASH = hashlib.sha256(manifest_text.encode()).hexdigest()
test_ids = sorted(p.name for p in (DATA_DIR / "test").glob("*.jpg"))
if len(paths) != EXPECTED_TRAIN or len(test_ids) != EXPECTED_TEST:
    raise ValueError(f"样本数量与本实验不符：训练 {len(paths)}，测试 {len(test_ids)}。请检查数据。")
template = None
for folder in [DATA_DIR, *list(DATA_DIR.parents)[:3]]:
    for name in ["sample_submission.csv", "submission.csv"]:
        candidate = folder / name
        if candidate.is_file():
            frame = pd.read_csv(candidate, dtype={"ID": str})
            if (list(frame.columns) == ["ID", "Emotion"] and frame.ID.is_unique
                    and len(frame) == len(test_ids) and set(frame.ID) == set(test_ids)):
                template = frame
                break
    if template is not None:
        test_ids = template.ID.tolist()
        break
print(pd.Series(np.bincount(labels, minlength=len(CLASSES)), index=CLASSES, name="全量训练图片数"))
print(f"训练 {len(paths)} | 验证 0 | 测试 {len(test_ids)}")
assert len(paths) == len(labels) == len(manifest)


## 2. 图像处理与模型
沿用验证实验的增强；六类内部索引顺序保持不变，官方数字编号只在导出提交时使用。


In [ ]:
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(10, translate=(0.05, 0.05), scale=(0.95, 1.05),
                            interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(), normalize,
])
eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(), normalize,
])

class FaceDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("RGB")
        return self.transform(image), int(self.labels[index])


def seed_worker(_):
    seed = torch.initial_seed() % (2 ** 32)
    random.seed(seed)
    np.random.seed(seed)


def loader(paths, labels, transform, shuffle=False):
    return DataLoader(FaceDataset(paths, labels, transform), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=WORKERS, pin_memory=device.type == "cuda",
                      persistent_workers=WORKERS > 0, worker_init_fn=seed_worker,
                      generator=torch.Generator().manual_seed(SEED))





def progress(data_loader, stage):
    started = time.monotonic()
    total = len(data_loader)
    print(f"{stage}开始，共 {total} 批", flush=True)
    for step, batch in enumerate(data_loader, 1):
        yield batch
        if step == 1 or step % 20 == 0 or step == total:
            elapsed = time.monotonic() - started
            print(f"  {stage} {step}/{total} 批 | 已用 {elapsed:.0f}s", flush=True)


@torch.inference_mode()
def predict(model, data_loader, flip=False, stage="预测"):
    model.eval()
    base, tta = [], []
    for x, _ in progress(data_loader, stage):
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(x)
            flipped = model(x.flip(-1)) if flip else None
        prob = logits.float().softmax(1)
        base.append(prob.cpu().numpy())
        tta.append(((prob + flipped.float().softmax(1)) / 2 if flip else prob).cpu().numpy())
    return np.concatenate(base), np.concatenate(tta)



def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def model_factory(architecture):
    if architecture == "resnet18":
        return models.resnet18, models.ResNet18_Weights.DEFAULT
    if architecture == "resnet34":
        return models.resnet34, models.ResNet34_Weights.DEFAULT
    raise ValueError(f"不支持的模型：{architecture}")


def build_model(dropout, pretrained=False, architecture="resnet18"):
    factory, weights = model_factory(architecture)
    model = factory(weights=weights if pretrained else None)
    model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.fc.in_features, len(CLASSES)))
    if architecture == "resnet34":
        # 两模型分类头维度相同。复现旧 ResNet18 的初始化，不读取已训练的头。
        with torch.random.fork_rng():
            torch.manual_seed(SEED)
            reference = models.resnet18(weights=None)
            head = nn.Linear(reference.fc.in_features, len(CLASSES))
            model.fc[1].load_state_dict(head.state_dict())
        del reference, head
    return model.to(device)


def parameter_count(architecture):
    factory, _ = model_factory(architecture)
    with torch.device("meta"):
        model = factory(weights=None)
        model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    return sum(p.numel() for p in model.parameters())



## 3. 全量训练：固定轮次，按轮续训
不计算验证指标，不保存“最佳验证模型”。`last.pt` 是最近完整轮次；训练完成后额外保存仅供预测的 `final.pt`。
程序仅在配置与数据指纹一致时恢复。多个匹配备份请通过 `RESUME_DIR` 指定；已有本地 checkpoint 优先，不会用旧备份覆盖。


In [ ]:
CONFIG = dict(pipeline="fulltrain_resnet34_v1", model="resnet34", seed=SEED,
              epochs=EPOCHS, scheduler_t_max=SCHEDULER_T_MAX, batch_size=BATCH_SIZE,
              image_size=IMAGE_SIZE, lr=LR, weight_decay=WEIGHT_DECAY, dropout=0.0,
              label_smoothing=0.0, workers=WORKERS, pretrained=PRETRAINED,
              classes=CLASSES, label_map=LABEL_MAP, data_hash=DATA_HASH, train_count=len(paths),
              torch_version=str(torch.__version__), torchvision_version=__import__("torchvision").__version__)


def atomic_save(obj, target):
    temporary = target.with_suffix(target.suffix + ".tmp")
    torch.save(obj, temporary)
    temporary.replace(target)


def prepare_fulltrain_output():
    config_path = OUT / "config.json"
    if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
        raise ValueError("OUT 已有不同配置，请使用新的输出目录，不要覆盖已有实验。")
    if not (OUT / "last.pt").exists():
        if RESUME_DIR is not None:
            sources = [Path(RESUME_DIR)]
        else:
            sources, incompatible = [], []
            for checkpoint_path in ROOT.rglob("last.pt"):
                parent = checkpoint_path.parent
                if parent.resolve() == OUT.resolve() or not (parent / "config.json").is_file():
                    continue
                saved_config = json.loads((parent / "config.json").read_text())
                if saved_config == CONFIG:
                    sources.append(parent)
                elif saved_config.get("pipeline") == CONFIG["pipeline"]:
                    incompatible.append(str(parent))
            if not sources and incompatible:
                raise ValueError("发现全量备份但配置或数据指纹不匹配，不会静默从头训练：" + str(incompatible))
        if len(sources) > 1:
            raise ValueError("存在多份匹配的全量训练备份，请设置 RESUME_DIR。")
        if sources:
            source = sources[0]
            if not (source / "last.pt").is_file() or json.loads((source / "config.json").read_text()) != CONFIG:
                raise ValueError("续训备份缺失 last.pt 或配置不同。")
            shutil.copy2(source / "last.pt", OUT / "last.restore.tmp")
            (OUT / "last.restore.tmp").replace(OUT / "last.pt")
            print("已恢复全量训练 checkpoint：", source, flush=True)
    if not config_path.exists():
        config_path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
    (OUT / "train_manifest.csv").write_text(manifest_text, encoding="utf-8")


def train_full():
    prepare_fulltrain_output()
    checkpoint_path = OUT / "last.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True) if checkpoint_path.exists() else None
    if checkpoint is not None and checkpoint["config"] != CONFIG:
        raise ValueError("checkpoint 内的配置不一致，停止恢复。")
    seed_all(SEED)
    model = build_model(0.0, pretrained=PRETRAINED if checkpoint is None else False, architecture="resnet34")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SCHEDULER_T_MAX)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    criterion = nn.CrossEntropyLoss()
    history, completed, previous_seconds = [], 0, 0.0
    if checkpoint is not None:
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        history = checkpoint["history"]
        completed = checkpoint["epoch"]
        previous_seconds = checkpoint["training_seconds"]
        if len(history) != completed or not 0 <= completed <= EPOCHS:
            raise ValueError("checkpoint 轮次记录异常。")
        if completed == EPOCHS:
            print(f"已完成全部 {EPOCHS} 轮，跳过训练，恢复最终模型。", flush=True)
        else:
            print(f"已恢复完整第 {completed} 轮，将从第 {completed+1} 轮继续；目标 {EPOCHS} 轮。", flush=True)
        del checkpoint
    full_loader = loader(paths, labels, train_tf, shuffle=True)
    assert len(full_loader.dataset) == len(paths)
    print(f"全量训练 {len(paths)} 张；每轮 {len(full_loader)} 批；参数量 {sum(p.numel() for p in model.parameters()):,}", flush=True)
    started = time.monotonic()
    for epoch in range(completed + 1, EPOCHS + 1):
        seed_all(SEED + epoch)
        full_loader.generator.manual_seed(SEED + epoch)
        model.train()
        loss_sum, correct, total, updates = 0.0, 0, 0, 0
        epoch_lr = optimizer.param_groups[0]["lr"]
        for x, y in progress(full_loader, f"全量 Epoch {epoch:02d}"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            if not torch.isfinite(loss):
                raise RuntimeError("训练损失非有限值，保留上一轮 checkpoint。")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            updates += int(scaler.get_scale() >= old_scale)
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            total += len(y)
        if total != len(paths) or updates == 0:
            raise RuntimeError("本轮未遍历全体训练图片或没有有效参数更新，保留上一轮 checkpoint。")
        history.append(dict(epoch=epoch, train_loss=loss_sum/total, train_accuracy=correct/total,
                            learning_rate=epoch_lr, images_seen=total, optimizer_updates=updates))
        scheduler.step()
        elapsed = previous_seconds + time.monotonic() - started
        state = dict(config=CONFIG, epoch=epoch, model=model.state_dict(), optimizer=optimizer.state_dict(),
                     scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), history=history,
                     training_seconds=elapsed)
        atomic_save(state, checkpoint_path)
        pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
        print(f"Epoch {epoch:02d}/{EPOCHS}: train_loss={loss_sum/total:.4f} train_acc={correct/total:.4f} "
              f"lr={epoch_lr:.7f} | checkpoint 已保存", flush=True)
    pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
    atomic_save(dict(config=CONFIG, epoch=EPOCHS, model=model.state_dict()), OUT / "final.pt")
    summary = dict(config=CONFIG, completed_epochs=EPOCHS, validation_used=False,
                   train_count=len(paths), parameters=sum(p.numel() for p in model.parameters()),
                   training_minutes=(previous_seconds + time.monotonic()-started)/60,
                   device_name=torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU")
    (OUT / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("全量训练完成；没有验证集指标。最终模型：", OUT / "final.pt", flush=True)
    del optimizer, scheduler, scaler, full_loader
    return model


model = train_full()


## 4. 水平翻转 TTA 与官方提交文件
TTA 已由此前验证实验选定，本轮不根据测试标签调参。输出每张测试图片的概率、英文类别和正确数字编码。


In [ ]:
checkpoint = torch.load(OUT / "final.pt", map_location="cpu", weights_only=True)
if checkpoint["config"] != CONFIG or checkpoint["epoch"] != EPOCHS:
    raise ValueError("最终模型配置或轮次不符。")
model.load_state_dict(checkpoint["model"])
del checkpoint
test_loader = loader([DATA_DIR / "test" / name for name in test_ids],
                     np.zeros(len(test_ids), dtype=np.int64), eval_tf)
_, test_prob = predict(model, test_loader, flip=True, stage="全量模型测试集 TTA")
if len(test_prob) != len(test_ids) or not np.isfinite(test_prob).all() or not np.allclose(test_prob.sum(1), 1, atol=1e-5):
    raise ValueError("测试预测数量或概率异常。")
probabilities = pd.DataFrame(test_prob, columns=CLASSES)
probabilities.insert(0, "ID", test_ids)
probabilities.to_csv(OUT / "test_probabilities.csv", index=False)
predictions = pd.DataFrame({"ID": test_ids, "EmotionName": [CLASSES[i] for i in test_prob.argmax(1)]})
predictions.to_csv(OUT / "predictions_labels.csv", index=False)
submission = pd.DataFrame({"ID": test_ids, "Emotion": predictions.EmotionName.map(LABEL_MAP)})
if submission.isna().any().any() or not submission.ID.is_unique or not submission.Emotion.isin(range(6)).all():
    raise ValueError("提交结果包含空值、重复 ID 或未知类别。")
submission.Emotion = submission.Emotion.astype("int64")
submission_path = OUT / "submission_fulltrain.csv"
submission.to_csv(submission_path, index=False)
display(submission.head())
print("全量训练提交文件：", submission_path, "| 行数：", len(submission))
display(FileLink(str(submission_path.relative_to(Path.cwd())) if submission_path.is_relative_to(Path.cwd()) else str(submission_path)))


## 5. 保存全量训练备份
在训练完成后，或手动停止训练后运行。至少完成一轮才有可恢复的 checkpoint。下载 ZIP；换会话时将其上传为私有 Dataset 并添加为 Input。恢复会读取最后完整轮次及优化器状态，不依赖最终模型文件是否已经生成。


In [ ]:
assert (OUT / "last.pt").exists(), "还没有完整轮次的 checkpoint，无法生成可续训备份"
archive_base = OUT.parent / "task2_fulltrain_backup"
shutil.make_archive(str(archive_base), "zip", root_dir=OUT.parent, base_dir=OUT.name)
archive_path = archive_base.with_suffix(".zip")
print("备份已生成，请下载：", archive_path)
display(FileLink(str(archive_path.relative_to(Path.cwd())) if archive_path.is_relative_to(Path.cwd()) else str(archive_path)))
